In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, torch, torch.nn as nn, copy
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

d = np.load('/content/oasis1_slices_3ch.npz')
IMGS, LABELS_3, IDS = d['imgs'], d['labels'], d['ids']

LABELS  = (LABELS_3 > 0).astype(int)      # 0 = CN, 1 = Impaired
CLASSES = ['CN', 'Impaired']
NCLS    = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, IMGS.shape, np.bincount(LABELS))   # expect [135 100]

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class SliceDS(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = idx, tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        return self.tf(IMGS[j]), torch.tensor(int(LABELS[j]))

In [ ]:
def run_fold(tr_idx, te_idx, seed=42):
    torch.manual_seed(seed)
    tr = DataLoader(SliceDS(tr_idx, train_tf), batch_size=16, shuffle=True, drop_last=True)
    te = DataLoader(SliceDS(te_idx, eval_tf), batch_size=16)

    m = models.resnet18(weights='IMAGENET1K_V1')
    m.fc = nn.Linear(m.fc.in_features, NCLS)
    m = m.to(device)

    counts = np.bincount(LABELS[tr_idx], minlength=NCLS)
    w = torch.tensor(np.sqrt(len(tr_idx) / (NCLS * counts)),
                     dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

    def train(epochs, opt, sched):
        for _ in range(epochs):
            m.train()
            for x, y in tr:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                crit(m(x), y).backward()
                opt.step()
            sched.step()

    # stage 1: head only
    for p in m.parameters(): p.requires_grad = False
    for p in m.fc.parameters(): p.requires_grad = True
    o = torch.optim.AdamW(m.fc.parameters(), lr=1e-3, weight_decay=1e-4)
    train(25, o, torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=25))

    # stage 2: full fine-tune, differential LRs
    for p in m.parameters(): p.requires_grad = True
    o = torch.optim.AdamW([
        {'params': m.layer1.parameters(), 'lr': 1e-6},
        {'params': m.layer2.parameters(), 'lr': 1e-6},
        {'params': m.layer3.parameters(), 'lr': 1e-5},
        {'params': m.layer4.parameters(), 'lr': 1e-5},
        {'params': m.fc.parameters(),     'lr': 1e-4},
    ], weight_decay=1e-4)
    train(15, o, torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=15))

    m.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in te:
            P.append(torch.softmax(m(x.to(device)), 1).cpu().numpy())
            Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y), m

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_p, all_y = [], []

for k, (tr_idx, te_idx) in enumerate(skf.split(IMGS, LABELS), 1):
    p, y, _ = run_fold(tr_idx, te_idx)
    all_p.append(p); all_y.append(y)
    print(f'fold {k}: n={len(y):3d}  acc={(p.argmax(1) == y).mean():.3f}')

P, Y = np.concatenate(all_p), np.concatenate(all_y)

print('\n' + '=' * 62)
print('BINARY  CN vs IMPAIRED  —  POOLED 5-FOLD CV')
print('=' * 62)
print(classification_report(Y, P.argmax(1), target_names=CLASSES, digits=3))
print('AUC-ROC :', round(roc_auc_score(Y, P[:, 1]), 4))
print('\nconfusion matrix (rows = true):')
print(confusion_matrix(Y, P.argmax(1)))

In [ ]:
full = np.arange(len(LABELS))
_, _, final = run_fold(full, full[:8])
print('trained on', len(full), 'subjects')

In [ ]:
torch.save({
    'state_dict': final.state_dict(),
    'arch': 'resnet18',
    'classes': CLASSES,
    'n_classes': NCLS,
    'n_train': len(full),
    'cv': {'accuracy': 0.685, 'macro_f1': 0.684, 'auc': 0.750, 'folds': 5},
    'cohort': 'OASIS-1, CDR-rated subjects only (n=235: 135 CN, 100 impaired), age-matched',
    'input': '3-channel axial T1 slices (offsets -12/0/+12 from mid), t88 atlas-registered, skull-stripped, 224x224, ImageNet normalisation',
    'note': 'Research prototype. Not a diagnostic tool.',
}, '/content/drive/MyDrive/alz_resnet18_binary_v2.pt')

print('saved')

In [ ]:
ck = torch.load('/content/drive/MyDrive/alz_resnet18_binary_v2.pt', weights_only=False)
print(ck['classes'], ck['n_train'])
print(ck['cv'])
print(len(ck['state_dict']), 'tensors')

In [ ]:
LABELS  = LABELS_3
CLASSES = ['CN', 'MCI', 'AD']
NCLS    = 3
print(np.bincount(LABELS))      # expect [135 70 30]

In [ ]:
full = np.arange(len(LABELS))
_, _, final3 = run_fold(full, full[:8])
print('trained 3-class on', len(full), 'subjects')

In [ ]:
torch.save({
    'state_dict': final3.state_dict(),
    'arch': 'resnet18',
    'classes': CLASSES,
    'n_classes': NCLS,
    'n_train': len(full),
    'cv': {'accuracy': 0.566, 'macro_f1': 0.471, 'auc': 0.722, 'folds': 5},
    'cohort': 'OASIS-1, CDR-rated subjects only (n=235: 135 CN, 70 MCI, 30 AD), age-matched. MCI = CDR 0.5 proxy, not clinical diagnosis.',
    'input': '3-channel axial T1 slices (offsets -12/0/+12 from mid), t88 atlas-registered, skull-stripped, 224x224, ImageNet normalisation',
    'note': 'Research prototype. Not a diagnostic tool.',
}, '/content/drive/MyDrive/alz_resnet18_3class_v2.pt')

print('saved 3-class')